In [ ]:
# # pip install dask dask-ml scikit-learn joblib
# !pip install dask-ml
# # pip install --upgrade dask-expr
# pip install "bokeh!=3.0.*,>=2.4.2"
# %pip install boto3
# import boto3

In [2]:
import dask
from dask.distributed import Client
client = Client()
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 11
Total threads: 11,Total memory: 18.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:51508,Workers: 11
Dashboard: http://127.0.0.1:8787/status,Total threads: 11
Started: Just now,Total memory: 18.00 GiB
Comm: tcp://127.0.0.1:51533,Total threads: 1
Dashboard: http://127.0.0.1:51538/status,Memory: 1.64 GiB
Nanny: tcp://127.0.0.1:51511,


In [3]:
client.dashboard_link

'http://127.0.0.1:8787/status'

In [3]:
import os
print(os.path.exists("final_joined_table.csv"))

True


## Random Forest Grid Search

In [8]:
import dask.dataframe as dd
from dask_ml.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.metrics import roc_auc_score, make_scorer
from sklearn.model_selection import train_test_split
from dask.distributed import Client
import pandas as pd
import boto3
import io
import time

# Step 2: Load local CSV
df_raw = pd.read_csv("final_joined_table.csv")

# Step 3: Preprocess
df_raw['emotion_clean'] = df_raw['emotion'].str.replace(r"[\[\]']", "", regex=True)
emotion_labels = ["joy", "sadness", "anger", "fear", "surprise", "disgust"]
features = ['acousticness', 'danceability', 'energy', 'instrumentalness',
            'liveness', 'loudness', 'speechiness', 'valence', 'tempo']

# Step 4: Grid Search Function with Expanded Param Grid
def run_rf_grid_dask(emotion_label):
    df = df_raw.copy()
    df['label'] = df['emotion_clean'].str.contains(emotion_label).astype(int)
    df = df.dropna(subset=features + ['label'])

    X_train, X_test, y_train, y_test = train_test_split(df[features], df['label'], test_size=0.2, random_state=42)

    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", RandomForestClassifier(random_state=42))  # n_jobs will be forced to 1 in Dask worker
    ])

    param_grid = {
        "clf__n_estimators": [100, 200],
        "clf__max_depth": [5, 10, 20],
        "clf__min_samples_leaf": [1, 2]
    }

    gs = GridSearchCV(pipeline, param_grid, scoring='roc_auc', cv=3)

    start = time.time()
    gs.fit(X_train, y_train)
    duration = time.time() - start

    y_pred_prob = gs.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_pred_prob)

    print(f"{emotion_label.upper()} done: AUC = {auc:.4f}, time = {duration:.2f}s")

    return {
        "Platform": "Dask",
        "Method": "Grid Search",
        "Model": "Random Forest",
        "Label": emotion_label,
        "Runtime (s)": round(duration, 2),
        "Best Score": round(auc, 4),
        "Resources Used": "Dask cluster (default)",
        "Notes": str(gs.best_params_)
    }

# Step 5: Run for all emotions
results = [run_rf_grid_dask(emotion) for emotion in emotion_labels]

# Step 6: Save to CSV
df_results = pd.DataFrame(results)
df_results.to_csv("/tmp/dask_rf_gridsearch_results.csv", index=False)
print("Saved to /tmp/dask_rf_gridsearch_results.csv")

JOY done: AUC = 0.5231, time = 5.25s
SADNESS done: AUC = 0.5967, time = 2.47s
ANGER done: AUC = 0.6290, time = 2.57s
FEAR done: AUC = 0.5556, time = 2.38s
SURPRISE done: AUC = 0.5207, time = 2.55s
DISGUST done: AUC = 0.6419, time = 3.69s
Saved to /tmp/dask_rf_gridsearch_results.csv


## Random Forest Ranom Search

In [10]:
import dask
from dask.distributed import Client
from dask_ml.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
import pandas as pd
import time

# Load preprocessed CSV (already cleaned)
df_raw = pd.read_csv("final_joined_table.csv")
df_raw['emotion_clean'] = df_raw['emotion'].str.replace(r"[\[\]']", "", regex=True)

# Emotion categories
emotion_labels = ["joy", "sadness", "anger", "fear", "surprise", "disgust"]

# Audio features
features = ['acousticness', 'danceability', 'energy', 'instrumentalness',
            'liveness', 'loudness', 'speechiness', 'valence', 'tempo']

# Track already-completed ones to skip
already_done = set()  

# Random Search function
def run_rf_random_dask(emotion_label):
    if emotion_label in already_done:
        print(f"{emotion_label.upper()} skipped.")
        return None

    df = df_raw.copy()
    df['label'] = df['emotion_clean'].str.contains(emotion_label).astype(int)
    df = df.dropna(subset=features + ['label'])

    X_train, X_test, y_train, y_test = train_test_split(
        df[features], df['label'], test_size=0.2, random_state=42
    )

    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", RandomForestClassifier(random_state=42))  # n_jobs will be ignored inside Dask workers
    ])

    param_dist = {
        "clf__n_estimators": [100, 200],
        "clf__max_depth": [5, 10, 20],
        "clf__min_samples_leaf": [1, 2]
    }

    search = RandomizedSearchCV(
        pipeline,
        param_distributions=param_dist,
        n_iter=5,  # number of random combinations to try
        scoring='roc_auc',
        cv=3,
        random_state=42
    )

    start = time.time()
    search.fit(X_train, y_train)
    duration = time.time() - start

    y_pred_prob = search.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_pred_prob)

    print(f"{emotion_label.upper()} done (Random Search): AUC = {auc:.4f}, time = {duration:.2f}s")

    return {
        "Platform": "Dask",
        "Method": "Random Search",
        "Model": "Random Forest",
        "Label": emotion_label,
        "Runtime (s)": round(duration, 2),
        "Best Score": round(auc, 4),
        "Resources Used": "Dask cluster (default)",
        "Notes": str(search.best_params_)
    }

# Run for each emotion
results = [run_rf_random_dask(e) for e in emotion_labels]
results = [r for r in results if r is not None]  # remove skipped

# Save to CSV
df_results = pd.DataFrame(results)
df_results.to_csv("/tmp/dask_rf_randomsearch_results.csv", index=False)
print("Saved to /tmp/dask_rf_randomsearch_results.csv")


JOY done (Random Search): AUC = 0.5231, time = 2.05s
SADNESS done (Random Search): AUC = 0.5980, time = 1.39s
ANGER done (Random Search): AUC = 0.6263, time = 1.49s
FEAR done (Random Search): AUC = 0.5556, time = 1.32s
SURPRISE done (Random Search): AUC = 0.5111, time = 1.40s
DISGUST done (Random Search): AUC = 0.6362, time = 1.22s
Saved to /tmp/dask_rf_randomsearch_results.csv


# MLP

In [ ]:
# MLP Grid Search aligned with Spark

from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from dask_ml.model_selection import GridSearchCV
from dask.distributed import Client
import pandas as pd
import time

# Step 1: Load and preprocess data
df_raw = pd.read_csv("final_joined_table.csv")
df_raw['emotion_clean'] = df_raw['emotion'].str.replace(r"[\[\]']", "", regex=True)

emotion_labels = ["joy", "sadness", "anger", "fear", "surprise", "disgust"]
features = ['acousticness', 'danceability', 'energy', 'instrumentalness',
            'liveness', 'loudness', 'speechiness', 'valence', 'tempo']

# Step 2: Run MLP grid search (aligned with Spark layers)
def run_mlp_grid_dask(emotion_label):
    df = df_raw.copy()
    df['label'] = df['emotion_clean'].str.contains(emotion_label).astype(int)
    df = df.dropna(subset=features + ['label'])

    X_train, X_test, y_train, y_test = train_test_split(
        df[features], df['label'], test_size=0.2, random_state=42
    )

    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("mlp", MLPClassifier(random_state=42, max_iter=150))  # match Spark's maxIter=150
    ])

    # Match Spark layers exactly
    param_grid = {
        "mlp__hidden_layer_sizes": [
            (100,),
            (100, 50),
            (128, 64, 32)
        ]
    }

    grid = GridSearchCV(
        pipeline,
        param_grid=param_grid,
        scoring='roc_auc',
        cv=2  # match Spark's numFolds=2
    )

    start = time.time()
    grid.fit(X_train, y_train)
    duration = time.time() - start

    y_pred_prob = grid.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_pred_prob)

    print(f"{emotion_label.upper()} done (MLP Grid): AUC = {auc:.4f}, time = {duration:.2f}s")

    return {
        "Platform": "Dask",
        "Method": "Grid Search",
        "Model": "MLP",
        "Label": emotion_label,
        "Runtime (s)": round(duration, 2),
        "Best Score": round(auc, 4),
        "Resources Used": "Dask cluster (default)",
        "Notes": str(grid.best_params_)
    }

# Step 3: Run for all emotion labels and save
results = [run_mlp_grid_dask(emotion) for emotion in emotion_labels]
df_result = pd.DataFrame(results)
df_result.to_csv("/tmp/dask_mlp_gridsearch_results.csv", index=False)
print("Saved to /tmp/dask_mlp_gridsearch_results_maxiter150.csv")

/Users/vera/Library/Python/3.9/lib/python/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (150) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/vera/Library/Python/3.9/lib/python/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (150) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/vera/Library/Python/3.9/lib/python/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (150) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/vera/Library/Python/3.9/lib/python/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (150) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/vera/Library/

JOY done (MLP Grid): AUC = 0.5784, time = 1.66s


/Users/vera/Library/Python/3.9/lib/python/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (150) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/vera/Library/Python/3.9/lib/python/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (150) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/vera/Library/Python/3.9/lib/python/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (150) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/vera/Library/Python/3.9/lib/python/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (150) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/vera/Library/

SADNESS done (MLP Grid): AUC = 0.6105, time = 1.16s


/Users/vera/Library/Python/3.9/lib/python/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (150) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/vera/Library/Python/3.9/lib/python/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (150) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/vera/Library/Python/3.9/lib/python/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (150) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/vera/Library/Python/3.9/lib/python/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (150) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/vera/Library/

...

/Users/vera/Library/Python/3.9/lib/python/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (150) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/vera/Library/Python/3.9/lib/python/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (150) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/vera/Library/Python/3.9/lib/python/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (150) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/vera/Library/Python/3.9/lib/python/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (150) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/vera/Library/

SURPRISE done (MLP Grid): AUC = 0.5339, time = 1.22s


/Users/vera/Library/Python/3.9/lib/python/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (150) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/vera/Library/Python/3.9/lib/python/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (150) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/vera/Library/Python/3.9/lib/python/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (150) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/vera/Library/Python/3.9/lib/python/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (150) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/vera/Library/

DISGUST done (MLP Grid): AUC = 0.6109, time = 1.29s
Saved to /tmp/dask_mlp_gridsearch_results.csv


/Users/vera/Library/Python/3.9/lib/python/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (150) reached and the optimization hasn't converged yet.
  warnings.warn(


In [15]:
# MLP Grid Search aligned with Spark

from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from dask_ml.model_selection import GridSearchCV
from dask.distributed import Client
import pandas as pd
import time

# Step 1: Load and preprocess data
df_raw = pd.read_csv("final_joined_table.csv")
df_raw['emotion_clean'] = df_raw['emotion'].str.replace(r"[\[\]']", "", regex=True)

emotion_labels = ["joy", "sadness", "anger", "fear", "surprise", "disgust"]
features = ['acousticness', 'danceability', 'energy', 'instrumentalness',
            'liveness', 'loudness', 'speechiness', 'valence', 'tempo']

# Step 2: Run MLP grid search (aligned with Spark layers)
def run_mlp_grid_dask(emotion_label):
    df = df_raw.copy()
    df['label'] = df['emotion_clean'].str.contains(emotion_label).astype(int)
    df = df.dropna(subset=features + ['label'])

    X_train, X_test, y_train, y_test = train_test_split(
        df[features], df['label'], test_size=0.2, random_state=42
    )

    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("mlp", MLPClassifier(random_state=42, max_iter=500, early_stopping=True, validation_fraction=0.1))
    ])

    # Match Spark layers exactly
    param_grid = {
        "mlp__hidden_layer_sizes": [
            (100,),
            (100, 50),
            (128, 64, 32)
        ]
    }

    grid = GridSearchCV(
        pipeline,
        param_grid=param_grid,
        scoring='roc_auc',
        cv=2  # match Spark's numFolds=2
    )

    start = time.time()
    grid.fit(X_train, y_train)
    duration = time.time() - start

    y_pred_prob = grid.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_pred_prob)

    print(f"{emotion_label.upper()} done (MLP Grid): AUC = {auc:.4f}, time = {duration:.2f}s")

    return {
        "Platform": "Dask",
        "Method": "Grid Search",
        "Model": "MLP",
        "Label": emotion_label,
        "Runtime (s)": round(duration, 2),
        "Best Score": round(auc, 4),
        "Resources Used": "Dask cluster (default)",
        "Notes": str(grid.best_params_)
    }

# Step 3: Run for all emotion labels and save
results = [run_mlp_grid_dask(emotion) for emotion in emotion_labels]
df_result = pd.DataFrame(results)
df_result.to_csv("/tmp/dask_mlp_gridsearch_results.csv", index=False)
print("Saved to /tmp/dask_mlp_gridsearch_results.csv")

JOY done (MLP Grid): AUC = 0.5521, time = 0.53s
SADNESS done (MLP Grid): AUC = 0.5942, time = 0.28s
ANGER done (MLP Grid): AUC = 0.5373, time = 0.32s
FEAR done (MLP Grid): AUC = 0.4855, time = 0.23s
SURPRISE done (MLP Grid): AUC = 0.5466, time = 0.42s
DISGUST done (MLP Grid): AUC = 0.5425, time = 0.21s
Saved to /tmp/dask_mlp_gridsearch_results.csv


In [16]:
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from dask_ml.model_selection import RandomizedSearchCV
from dask.distributed import Client
import pandas as pd
import time

# Load dataset
df_raw = pd.read_csv("final_joined_table.csv")
df_raw['emotion_clean'] = df_raw['emotion'].str.replace(r"[\[\]']", "", regex=True)

# Define emotions and audio features
emotion_labels = ["joy", "sadness", "anger", "fear", "surprise", "disgust"]
features = ['acousticness', 'danceability', 'energy', 'instrumentalness',
            'liveness', 'loudness', 'speechiness', 'valence', 'tempo']

# Randomized Search with 6 combinations
def run_mlp_random_dask_matched(emotion_label):
    df = df_raw.copy()
    df['label'] = df['emotion_clean'].str.contains(emotion_label).astype(int)
    df = df.dropna(subset=features + ['label'])

    X_train, X_test, y_train, y_test = train_test_split(
        df[features], df['label'], test_size=0.2, random_state=42
    )

    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", MLPClassifier(random_state=42, max_iter=500, early_stopping=True, validation_fraction=0.1))
    ])

    param_dist = {
        "clf__hidden_layer_sizes": [
            (64,), (128, 64), (32, 16), (50, 50)
        ],
        "clf__learning_rate_init": [0.01, 0.05],   # Spark stepSize ≈ learning_rate_init
        "clf__batch_size": [64, 128]              # Spark blockSize ≈ batch_size
    }

    search = RandomizedSearchCV(
        pipeline,
        param_distributions=param_dist,
        n_iter=6,  # Match Spark combo count
        scoring='roc_auc',
        cv=2,
        random_state=42
    )

    start = time.time()
    search.fit(X_train, y_train)
    duration = time.time() - start

    y_pred_prob = search.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_pred_prob)

    print(f"{emotion_label.upper()} done (MLP Random): AUC = {auc:.4f}, time = {duration:.2f}s")

    return {
        "Platform": "Dask",
        "Method": "Random Search",
        "Model": "MLP",
        "Label": emotion_label,
        "Runtime (s)": round(duration, 2),
        "Best Score": round(auc, 4),
        "Resources Used": "Dask cluster (default)",
        "Notes": str(search.best_params_)
    }

# Run search across emotions
results = [run_mlp_random_dask_matched(label) for label in emotion_labels]
df_result = pd.DataFrame(results)
df_result.to_csv("/tmp/dask_mlp_randomsearch_matched_results.csv", index=False)
print("Saved to /tmp/dask_mlp_randomsearch_matched_results.csv")


JOY done (MLP Random): AUC = 0.5780, time = 0.56s
SADNESS done (MLP Random): AUC = 0.5892, time = 0.31s
ANGER done (MLP Random): AUC = 0.5842, time = 0.35s
FEAR done (MLP Random): AUC = 0.4951, time = 0.35s
SURPRISE done (MLP Random): AUC = 0.5214, time = 0.28s
DISGUST done (MLP Random): AUC = 0.5995, time = 0.33s
Saved to /tmp/dask_mlp_randomsearch_matched_results.csv
